# Curriculum–Industry Skill Alignment using Feast

In [ ]:
!pip -q install feast==0.64.0 pyarrow

In [ ]:
import feast
print("Feast version:", feast.__version__)

**Upload the CSE Skill Alignment Dataset**

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
df = pd.read_excel("Curriculum_Industry_Skill_Alignment_Dataset.xlsx", sheet_name="Student_Skill_Data")
print(df.head())
print("Dataset shape:", df.shape)

**Examine the Data**

In [ ]:
print(df.columns.tolist())
df.info()
print("\nMissing Values:\n", df.isnull().sum())

**Create Features - skill alignment feature engineering**

In [ ]:
skills=["Programming","Database","Problem_Solving","Communication","Cloud_Computing","Data_Analysis","Teamwork","Aptitude"]
df=df.copy(); df["student_id"]=df.Student_ID.astype(str)
for c in skills:
    df[c]=pd.to_numeric(df[c],errors="coerce")
    df[c]=df[c].fillna(df[c].median())
df["technical_skill_score"]=df[["Programming","Database","Cloud_Computing","Data_Analysis"]].mean(axis=1).round(2)
df["professional_skill_score"]=df[["Problem_Solving","Communication","Teamwork","Aptitude"]].mean(axis=1).round(2)
df["calculated_skill_score"]=df[skills].mean(axis=1).round(2)
df["average_skill_gap"]=pd.to_numeric(df["Average_Skill_Gap"],errors="coerce")
df["skill_gap_category"]=df["Skill_Gap_Category"].astype(str)
df["recommended_training_area"]=df["Recommended_Training_Area"].astype(str)

In [ ]:
display(df[["student_id",*skills,"technical_skill_score","professional_skill_score","calculated_skill_score","average_skill_gap","skill_gap_category"]].head())

**Add Event Timestamps**

In [ ]:
base=pd.Timestamp("2025-01-01",tz="UTC")
n=df.student_id.str.extract(r"(\d+)",expand=False).astype(int)
df["event_timestamp"]=base+pd.to_timedelta(n,unit="D")
df["created_timestamp"]=df.event_timestamp+pd.Timedelta(hours=1)
print(df[["student_id","event_timestamp","created_timestamp"]].head())

**Separate Features and Labels**

In [ ]:
feature_df=df[["student_id","event_timestamp","created_timestamp",*skills,"technical_skill_score","professional_skill_score","calculated_skill_score"]].copy()
for c in skills: feature_df[c]=feature_df[c].round().astype("int64")
for c in ["technical_skill_score","professional_skill_score","calculated_skill_score"]: feature_df[c]=feature_df[c].astype("float32")

In [ ]:
label_df=df[["student_id","event_timestamp","skill_gap_category","average_skill_gap","recommended_training_area"]].copy()
print(label_df["skill_gap_category"].value_counts())

In [ ]:
display(feature_df.head()); display(label_df.head())

**Create the Feast Repository**

In [ ]:
import os
repo_path="/content/cse_skill_feast"
os.makedirs(f"{repo_path}/data",exist_ok=True)
print(repo_path)

In [ ]:
os.makedirs("/content/cse_skill_feast/data",exist_ok=True); print("Repository ready.")

In [ ]:
feature_df.to_parquet("/content/cse_skill_feast/data/cse_skill_features.parquet",index=False); print("Feature data saved.")

**Create feature_store.yaml**

In [ ]:
yaml="""project: cse_skill_project
registry: data/registry.db
provider: local
offline_store:
  type: file
online_store:
  type: sqlite
  path: data/online_store.db
"""
open("/content/cse_skill_feast/feature_store.yaml","w").write(yaml)
print("feature_store.yaml created.")

**Define the Entity and Feature View**

In [ ]:
definition="""from datetime import timedelta
from feast import Entity,FeatureView,FeatureService,Field,FileSource
from feast.types import Float32,Int64
student=Entity(name="student",join_keys=["student_id"],description="CSE student")
source=FileSource(name="cse_skill_source",path="data/cse_skill_features.parquet",timestamp_field="event_timestamp",created_timestamp_column="created_timestamp")
view=FeatureView(name="cse_skill_features",entities=[student],ttl=timedelta(days=3650),schema=[
Field(name="Programming",dtype=Int64),Field(name="Database",dtype=Int64),Field(name="Problem_Solving",dtype=Int64),Field(name="Communication",dtype=Int64),Field(name="Cloud_Computing",dtype=Int64),Field(name="Data_Analysis",dtype=Int64),Field(name="Teamwork",dtype=Int64),Field(name="Aptitude",dtype=Int64),Field(name="technical_skill_score",dtype=Float32),Field(name="professional_skill_score",dtype=Float32),Field(name="calculated_skill_score",dtype=Float32)],source=source,online=True)
service=FeatureService(name="cse_skill_gap_service",features=[view])
"""
open("/content/cse_skill_feast/features.py","w").write(definition)
print("features.py created.")

In [ ]:
!find /content/cse_skill_feast -maxdepth 2 -type f

**Apply the Feature Definitions**

In [ ]:
%cd /content/cse_skill_feast

In [ ]:
!feast apply

In [ ]:
!feast entities list

In [ ]:
!feast feature-views list

**Create the Feast Store Object**

In [ ]:
from feast import FeatureStore
store=FeatureStore(repo_path="/content/cse_skill_feast")
print("FeatureStore created.")

In [ ]:
feature_service=store.get_feature_service("cse_skill_gap_service")
print("Feature service loaded.")

**Retrieve Historical Features**

In [ ]:
entity_df=label_df[["student_id","event_timestamp","skill_gap_category","average_skill_gap","recommended_training_area"]].copy(); display(entity_df.head())

In [ ]:
training_data=store.get_historical_features(entity_df=entity_df,features=feature_service).to_df(); print("Historical retrieval completed.")

In [ ]:
display(training_data.head()); print("Training shape:",training_data.shape)

**Train a Model Using Feast Features**

In [ ]:
feature_columns=skills; print(feature_columns)

In [ ]:
X=training_data[feature_columns]; y=training_data["skill_gap_category"]; print(X.shape,y.shape); print(y.value_counts())

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
print(len(X_train),len(X_test))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
model=DecisionTreeClassifier(max_depth=4,random_state=42); model.fit(X_train,y_train); print("Model trained.")

In [ ]:
predictions=model.predict(X_test); print(predictions[:20])

In [ ]:
from sklearn.metrics import accuracy_score,classification_report
acc=accuracy_score(y_test,predictions); print("Accuracy:",round(acc*100,2),"%"); print(classification_report(y_test,predictions))

The model uses the eight CSE employability skill scores retrieved through Feast historical features.

**Materialize Features to the Online Store**

In [ ]:
%cd /content/cse_skill_feast
!feast materialize 2025-01-01T00:00:00 2025-05-01T00:00:00

**Retrieve Online Features**

In [ ]:
online=store.get_online_features(features=feature_service,entity_rows=[{"student_id":"CSE025"}]).to_dict()
print("Online features retrieved.")

In [ ]:
print(online)

In [ ]:
online_df=pd.DataFrame(online); display(online_df)

**Make an Online Skill-Gap Prediction**

In [ ]:
final_model=DecisionTreeClassifier(max_depth=4,random_state=42); final_model.fit(X,y); print("Final model trained.")

In [ ]:
X_online=online_df[feature_columns]; display(X_online)

In [ ]:
p=final_model.predict(X_online); print("Student:",online_df.student_id.iloc[0]); print("Predicted Skill Gap:",p[0])

**Retrieve Features for Multiple CSE Students**

In [ ]:
online=store.get_online_features(features=feature_service,entity_rows=[{"student_id":f"CSE{i:03d}"} for i in [10,20,30,40,50]]).to_dict()
online_df=pd.DataFrame(online); display(online_df)

In [ ]:
online_df["predicted_skill_gap"]=final_model.predict(online_df[feature_columns])
display(online_df[["student_id","predicted_skill_gap"]])